In [1]:
!pip install vllm -q

In [2]:
!nvidia-smi

Mon May 25 00:49:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from vllm import LLM, SamplingParams

llm = LLM(
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    enforce_eager=True  # disables CUDA graph — more stable on Colab
)

sampling_params = SamplingParams(temperature=0.8, max_tokens=100)

prompt = ["Explain what a KV cache is in 3 sentences."]
output = llm.generate(prompt, sampling_params)
print(output[0].outputs[0].text)

INFO 05-25 00:49:30 [utils.py:240] non-default args: {'disable_log_stats': True, 'enforce_eager': True, 'model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 05-25 00:49:31 [model.py:568] Resolved architecture: LlamaForCausalLM
WARNING 05-25 00:49:31 [model.py:1982] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 05-25 00:49:31 [model.py:2035] Casting torch.bfloat16 to torch.float16.
INFO 05-25 00:49:31 [model.py:1697] Using max model len 2048
INFO 05-25 00:49:31 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-25 00:49:31 [vllm.py:886] Asynchronous scheduling is enabled.
WARNING 05-25 00:49:31 [vllm.py:942] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-25 00:49:31 [vllm.py:960] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-25 00:49:31 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPrior

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


2. Explain how the KV cache works in detail, including how the cache actually stores the data, where the data is stored, and what kind of optimizations are possible.


In [4]:
import time

sampling_params = SamplingParams(temperature=0.8, max_tokens=100)
prompt = "Explain what a KV cache is in 3 sentences."

# Test 1: Single request
start = time.time()
llm.generate([prompt], sampling_params)
single = time.time() - start
print(f"1 request: {single:.2f}s")

# Test 2: 2 requests in one batch (how vLLM actually works)
start = time.time()
llm.generate([prompt, prompt], sampling_params)
two = time.time() - start
print(f"2 requests (batched): {two:.2f}s | per request: {two/2:.2f}s")

# Test 3: 4 requests in one batch
start = time.time()
llm.generate([prompt, prompt, prompt, prompt], sampling_params)
four = time.time() - start
print(f"4 requests (batched): {four:.2f}s | per request: {four/4:.2f}s")

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

1 request: 0.09s


Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

2 requests (batched): 1.38s | per request: 0.69s


Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

4 requests (batched): 2.23s | per request: 0.56s
